# Final Model Selection

Set **`CHOICE`** to the model key you picked (e.g. `'xgboost_model'`, `'random_forest'`, `'catboost_model'`). The model is built with its final hyperparameters (`src/models.py → FINAL_PARAMS`), fitted on the training split, evaluated once on validation + test, and saved to `models/final_model.pkl` + `models/preprocessor.pkl` for `predict_demo.ipynb` and `src/prediction.py`.

In [ ]:
CHOICE = None   # <- set to a model key, e.g. 'xgboost_model'

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root -> `import src`
warnings.filterwarnings('ignore')
from src.preprocessing import prepare_data, assert_run_consistency
from src.models import make_model, display_name, final_params, PARAM_SOURCE
from src.evaluation import evaluate, results_table, print_summary, save_results, plot_diagnostics
from src.prediction import save_final

In [ ]:
data = prepare_data()
n_rows = sum(data['meta']['rows'][s] for s in ('train', 'val', 'test'))
if CHOICE is None:
    raise ValueError('Set CHOICE to a model key first (see reports/final_evaluation_report.md).')

In [ ]:
model = make_model(CHOICE, data['target_transform'])
results, pred, _ = evaluate(model, data, display_name(CHOICE), CHOICE, evaluate_test=True)
print_summary(results, n_rows, len(data['feature_names']))
results_table(results)

In [ ]:
meta = {'model': display_name(CHOICE), 'key': CHOICE, 'hyperparameters': final_params(CHOICE),
        'param_source': PARAM_SOURCE[CHOICE], 'reference_id': data['fingerprint']['reference']['reference_id'],
        'cv_rmse': results['cv']['RMSE_mean'], 'val_rmse': results['val']['RMSE'], 'val_mae': results['val']['MAE'],
        'test_rmse': results['test']['RMSE'], 'test_mae': results['test']['MAE'], 'test_r2': results['test']['R2'],
        'test_mape': results['test']['MAPE'], 'target_transform': data['target_transform']}
save_final(model, data['preprocessor'], data['feature_names'], meta)
print('Saved models/final_model.pkl + models/preprocessor.pkl'); meta